<a href="https://colab.research.google.com/github/ms-solly/idbi-hackathon-2026/blob/main/notebooks/02_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Preprocessing

This notebook prepares the "Give Me Some Credit" dataset for machine learning.

The preprocessing pipeline includes:

- Removing unnecessary columns
- Splitting the dataset into training and testing sets
- Handling missing values
- Scaling numerical features
- Handling class imbalance using SMOTE

No models are trained in this notebook.

In [72]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from imblearn.over_sampling import SMOTE

from IPython.display import display

## Load Dataset

In [73]:
DATA_PATH = "cs-training.csv"

df = pd.read_csv(DATA_PATH)

print(df.shape)

display(df.head())

(62947, 12)


,Unnamed: 0,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
0,1,1.0,0.766127,45.0,2.0,0.802982,9120.0,13.0,0.0,6.0,0.0,2.0
1,2,0.0,0.957151,40.0,0.0,0.121876,2600.0,4.0,0.0,0.0,0.0,1.0
2,3,0.0,0.658180,38.0,1.0,0.085113,3042.0,2.0,1.0,0.0,0.0,0.0
3,4,0.0,0.233810,30.0,0.0,0.036050,3300.0,5.0,0.0,0.0,0.0,0.0
4,5,0.0,0.907239,49.0,1.0,0.024926,63588.0,7.0,0.0,1.0,0.0,0.0


## Remove Unnecessary Columns

In [74]:
df.drop(columns=["Unnamed: 0"], inplace=True)

print(df.shape)

(62947, 11)


In [75]:
# Remove the row with missing target
df = df.dropna(subset=["SeriousDlqin2yrs"])

# Verify
print(df["SeriousDlqin2yrs"].value_counts(dropna=False))

SeriousDlqin2yrs
0.0    58743
1.0     4203
Name: count, dtype: int64


## Separate Features and Target

In [76]:
X = df.drop("SeriousDlqin2yrs", axis=1)

y = df["SeriousDlqin2yrs"]

print(X.shape)
print(y.shape)

(62946, 10)
(62946,)


In [77]:
print(df["SeriousDlqin2yrs"].isnull().sum())
print(df["SeriousDlqin2yrs"].value_counts(dropna=False))

0
SeriousDlqin2yrs
0.0    58743
1.0     4203
Name: count, dtype: int64


## Train-Test Split

In [78]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Set :", X_train.shape)
print("Testing Set :", X_test.shape)

Training Set : (50356, 10)
Testing Set : (12590, 10)


## Handle Missing Values

In [79]:
print(X_train.isnull().sum())

RevolvingUtilizationOfUnsecuredLines       0
age                                        0
NumberOfTime30-59DaysPastDueNotWorse       0
DebtRatio                                  0
MonthlyIncome                           9957
NumberOfOpenCreditLinesAndLoans            0
NumberOfTimes90DaysLate                    0
NumberRealEstateLoansOrLines               0
NumberOfTime60-89DaysPastDueNotWorse       0
NumberOfDependents                      1326
dtype: int64


In [80]:
imputer = SimpleImputer(strategy="median")

X_train = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=X_train.columns
)

X_test = pd.DataFrame(
    imputer.transform(X_test),
    columns=X_test.columns
)

In [81]:
print(X_train.isnull().sum().sum())
print(X_test.isnull().sum().sum())

0
0


## Feature Scaling

In [82]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(
    X_train_scaled,
    columns=X_train.columns
)

X_test_scaled = pd.DataFrame(
    X_test_scaled,
    columns=X_test.columns
)

display(X_train_scaled.head())

In [83]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train_scaled,
    y_train
)

X_train_smote = pd.DataFrame(
    X_train_smote,
    columns=X_train.columns
)

y_train_smote = pd.Series(
    y_train_smote,
    name="SeriousDlqin2yrs"
)

print(y_train_smote.value_counts())

SeriousDlqin2yrs
0.0    46994
1.0    46994
Name: count, dtype: int64


In [84]:
X_test_scaled = pd.DataFrame(
    X_test_scaled,
    columns=X_train.columns
)

In [85]:
X_train_smote.reset_index(drop=True, inplace=True)
X_test_scaled.reset_index(drop=True, inplace=True)
y_train_smote.reset_index(drop=True, inplace=True)
y_test.reset_index(drop=True, inplace=True)

## Handle Class Imbalance using SMOTE

In [86]:
print(y_train.value_counts())
type(X_train_smote)
type(X_test_scaled)
type(y_train_smote)
type(y_test)

SeriousDlqin2yrs
0.0    46994
1.0     3362
Name: count, dtype: int64


pandas.core.series.Series

In [87]:
smote = SMOTE(random_state=42)

X_test_scaled = pd.DataFrame(
    X_test_scaled,
    columns=X_train.columns
)

In [88]:
print(y_train_smote.value_counts())

SeriousDlqin2yrs
0.0    46994
1.0    46994
Name: count, dtype: int64


In [89]:
import os

os.makedirs("../data/processed", exist_ok=True)

## Save Processed Dataset

In [90]:
X_train_smote.to_csv("../data/processed/X_train.csv", index=False)

X_test_scaled.to_csv("../data/processed/X_test.csv", index=False)

y_train_smote.to_csv("../data/processed/y_train.csv", index=False)

y_test.to_csv("../data/processed/y_test.csv", index=False)

print("Processed datasets saved successfully.")

Processed datasets saved successfully.


## Summary

The preprocessing pipeline has successfully:

- Removed unnecessary ID columns
- Split the dataset into training and testing sets
- Imputed missing values using median imputation
- Standardized numerical features using StandardScaler
- Balanced the training dataset using SMOTE

The processed datasets are now ready for model training.

In [92]:
import os
print(os.listdir())

['.config', 'cs-training.csv', 'sample_data']
